In [1]:
%pip install requests bs4 tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import time
import random
import pickle
from datetime import datetime

import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

import urllib3

In [ ]:
INPUT_FILE = "plants_with_taxonomy.csv"
OUTPUT_FILE = "wfo_descriptions.csv"
FAILED_FILE = "wfo_failed.csv"

CHECKPOINT_FILE = "wfo_scrape_checkpoint.pkl"

CHECKPOINT_EVERY = 50
SLEEP_RANGE = (5, 10)

MAX_CONSECUTIVE_BLOCK_ERRORS = 3
BATCH_SIZE = 100

In [4]:
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

retry_strategy = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=2,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504
    ],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(
    max_retries=retry_strategy
)

session = requests.Session()

session.mount(
    "http://",
    adapter
)

session.mount(
    "https://",
    adapter
)

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/130.0 Safari/537.36"
    )
})

In [5]:
def log(message):
    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )
    print(f"[{timestamp}] {message}")


In [6]:
def save_checkpoint(processed_ipni_ids):

    with open(
        CHECKPOINT_FILE,
        "wb"
    ) as f:

        pickle.dump(
            {
                "processed_ipni_ids": processed_ipni_ids
            },
            f
        )

    log(
        f"CHECKPOINT | "
        f"{len(processed_ipni_ids):,} plants saved"
    )


def load_checkpoint():

    if not os.path.exists(
        CHECKPOINT_FILE
    ):
        return set()

    with open(
        CHECKPOINT_FILE,
        "rb"
    ) as f:

        checkpoint = pickle.load(f)

    processed = checkpoint.get(
        "processed_ipni_ids",
        set()
    )

    log(
        f"RESUME | "
        f"{len(processed):,} plants already completed"
    )

    return processed



In [7]:
df = pd.read_csv(INPUT_FILE)

if "ipni_id" not in df.columns:
    raise ValueError(
        "ipni_id column not found"
    )

log(
    f"Loaded {len(df):,} rows"
)

[2026-10-23 13:25:06] Loaded 11,790 rows


In [8]:
unique_ipni_ids = (
    df["ipni_id"]
    .dropna()
    .astype(str)
    .unique()
)

log(
    f"Unique plants: "
    f"{len(unique_ipni_ids):,}"
)

processed_ipni_ids = load_checkpoint()

remaining_ids = [
    ipni_id
    for ipni_id in unique_ipni_ids
    if ipni_id not in processed_ipni_ids
]

log(
    f"{len(remaining_ids):,} "
    f"plants remaining"
)

[2026-10-23 13:25:06] Unique plants: 11,790
[2026-10-23 13:25:06] 11,790 plants remaining


In [9]:
records_buffer = []

def save_batch():

    global records_buffer

    if not records_buffer:
        return

    batch_df = pd.DataFrame(
        records_buffer
    )

    file_exists = os.path.exists(
        OUTPUT_FILE
    )

    batch_df.to_csv(
        OUTPUT_FILE,
        mode="a",
        header=not file_exists,
        index=False
    )

    log(
        f"SAVED "
        f"{len(batch_df):,} rows"
    )

    records_buffer = []

In [ ]:
success_count = len(
    processed_ipni_ids
)

failure_count = 0

processed_since_save = 0

start_time = time.time()

def plants_per_minute():

    elapsed_minutes = (
        time.time() - start_time
    ) / 60

    if elapsed_minutes == 0:
        return 0

    return success_count / elapsed_minutes


consecutive_block_errors = 0

BLOCK_INDICATORS = [
    "too many 500 error responses",
    "Max retries exceeded",
    "ResponseError",
    "SSLEOFError",
    "429",
    "Connection reset",
    "RemoteDisconnected",
    "Read timed out"
]

for ipni_id in tqdm(
    remaining_ids
):

    url = (
        f"https://www.worldfloraonline.org/"
        f"ipni/{ipni_id}"
    )

    try:

        response = session.get(
            url,
            timeout=30,
            verify=False
        )

        response.raise_for_status()

        consecutive_block_errors = 0

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        citation_lookup = {}

        bibliography = soup.find(
            "section",
            id="bibliography"
        )

        if bibliography:

            for li in bibliography.select(
                "ul > li"
            ):

                anchor = li.find(
                    "a",
                    id=True
                )

                if anchor:

                    citation_lookup[
                        anchor["id"]
                    ] = li.get_text(
                        " ",
                        strip=True
                    )

        found_description = False

        general = soup.find(
            "div",
            class_="tab-pane active"
        )

        if general:

            descriptions = general.find_all(
                "div",
                class_="description-with-citations"
            )

            for desc_idx, desc in enumerate(
                descriptions
            ):

                found_description = True

                header = None

                section_div = desc.parent

                h2 = section_div.find(
                    "h2"
                )

                if h2:

                    span = h2.find(
                        "span"
                    )

                    if span:

                        header = (
                            span.get_text(
                                strip=True
                            )
                        )

                    else:

                        header = (
                            h2.get_text(
                                " ",
                                strip=True
                            )
                        )

                if not header:

                    header = section_div.get(
                        "id"
                    )

                p = desc.find("p")

                description_text = (
                    p.get_text(
                        " ",
                        strip=True
                    )
                    if p
                    else None
                )

                provider = desc.find(
                    "li",
                    style=lambda s:
                    s and "float: right" in s
                )

                provider_text = (
                    provider.get_text(
                        " ",
                        strip=True
                    )
                    if provider
                    else None
                )

                citation_ids = []
                citation_texts = []

                citations_ul = desc.find(
                    "ul",
                    class_="citations"
                )

                if citations_ul:

                    for a in citations_ul.find_all(
                        "a"
                    ):

                        href = a.get(
                            "href"
                        )

                        if (
                            href
                            and href.startswith("#")
                        ):

                            citation_id = href[1:]

                            citation_ids.append(
                                citation_id
                            )

                            if (
                                citation_id
                                in citation_lookup
                            ):

                                citation_texts.append(
                                    citation_lookup[
                                        citation_id
                                    ]
                                )

                records_buffer.append(
                    {
                        "ipni_id":
                            ipni_id,
                        "description_id":
                            desc_idx,
                        "description_header":
                            header,
                        "description":
                            description_text,
                        "provider":
                            provider_text,
                        "citation_ids":
                            ";".join(
                                citation_ids
                            ),
                        "citations":
                            " || ".join(
                                citation_texts
                            ),
                        "url":
                            url
                    }
                )

        if not found_description:

            records_buffer.append({
                "ipni_id":
                    ipni_id,
                "description_id":
                    None,
                "description_header":
                    None,
                "description":
                    None,
                "provider":
                    None,
                "citation_ids":
                    None,
                "citations":
                    None,
                "url":
                    url
            })

        success_count += 1

        processed_ipni_ids.add(
            ipni_id
        )

        processed_since_save += 1

        if success_count % 25 == 0:

            log(
                f"PROGRESS | "
                f"success={success_count:,} | "
                f"failed={failure_count:,} | "
                f"rate={plants_per_minute():.1f} "
                f"plants/min"
            )

        if len(records_buffer) >= BATCH_SIZE:

            save_batch()

        if (
            processed_since_save
            >= CHECKPOINT_EVERY
        ):

            save_batch()

            save_checkpoint(
                processed_ipni_ids
            )

            processed_since_save = 0

        time.sleep(
            random.uniform(
                *SLEEP_RANGE
            )
        )

    except Exception as e:

        failure_count += 1

        error_msg = str(e)

        log(
            f"ERROR | "
            f"{ipni_id} | "
            f"{error_msg}"
        )

        pd.DataFrame(
            [{
                "ipni_id":
                    ipni_id,
                "error":
                    error_msg
            }]
        ).to_csv(
            FAILED_FILE,
            mode="a",
            header=not os.path.exists(
                FAILED_FILE
            ),
            index=False
        )

        is_block_error = any(
            indicator in error_msg
            for indicator in BLOCK_INDICATORS
        )

        if is_block_error:

            consecutive_block_errors += 1

            log(
                f"BLOCK WARNING | "
                f"{consecutive_block_errors}/"
                f"{MAX_CONSECUTIVE_BLOCK_ERRORS}"
            )

            if (
                consecutive_block_errors
                >= MAX_CONSECUTIVE_BLOCK_ERRORS
            ):

                log(
                    "LIKELY RATE LIMITED OR BLOCKED. "
                    "Saving checkpoint and exiting."
                )

                save_batch()

                save_checkpoint(
                    processed_ipni_ids
                )

                raise SystemExit(
                    "Stopped due to repeated "
                    "connection/block errors."
                )

            time.sleep(60)

        else:

            consecutive_block_errors = 0

        continue

save_batch()

save_checkpoint(
    processed_ipni_ids
)

elapsed = (
    time.time() - start_time
) / 60

log("FINISHED")

log(
    f"Success: "
    f"{success_count:,}"
)

log(
    f"Failed : "
    f"{failure_count:,}"
)

log(
    f"Elapsed : "
    f"{elapsed:.1f} min"
)

log(
    f"Rate    : "
    f"{plants_per_minute():.1f} "
    f"plants/min"
)

log(
    f"Output   : "
    f"{OUTPUT_FILE}"
)

  0%|          | 2/11790 [00:24<39:28:39, 12.06s/it]
